# Benchmarks per evaluation period

Reference figures every trial in `REGISTRY.md` is compared against. Nothing is
tuned here; the four benchmarks and the three periods are **frozen** in
`bt_periods.py`.

| Benchmark | Definition |
|---|---|
| `bh` | Buy and hold the S&P 500 total-return series |
| `sma200` | Long equities when close > SMA200 else T-bills; monthly check, next-close execution, 5 bp per switch |
| `mix60` | 60/40 equity / T-bill constant mix, monthly rebalance, no costs |
| `tbill` | 3-month T-bill (FRED `TB3MS`) |

Periods: **dev** 1934-02 to 1989-12, **val** 1990 to 2009, **holdout** 2010 onward
(the holdout is confirmatory only and never used for selection).

Primary metric (frozen): excess-return Sharpe net of costs, see `PLAN.md` §2.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_long as bl  # noqa: E402

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("periods:", {k: (str(v[0])[:10], str(v[1])[:10] if v[1] is not None else None) for k, v in bp.PERIODS.items()})

periods: {'dev': ('1934-02-01', '1989-12-31'), 'val': ('1990-01-01', '2009-12-31'), 'holdout': ('2010-01-01', None)}


## 1. Long sample (1934 onward)

In [2]:
S = bl.long_sample("1934-01-01")
px, cash = S["px"], S["cash"]
print(px.index[0].date(), "->", px.index[-1].date(), len(px), "days")
display(px.tail(3))

1934-02-01 -> 2026-10-19 23300 days


,close,tr_close
Date,,
2026-10-15,"7,744.0601","15,155.3393"
2026-10-16,"7,659.4399","14,989.8794"
2026-10-19,"7,700.4199","15,070.3769"


## 2. Benchmark returns and per-period statistics

In [3]:
bench = bp.benchmark_returns(px, cash)
tbl = bp.benchmark_table(bench)
tbl.index.names = ["period", "benchmark"]
display(tbl[["CAGR", "Vol", "Sharpe", "Max DD"]])

CAGR    Vol Sharpe  Max DD
period  benchmark                             
dev     bh        0.1106 0.1533 0.5110 -0.5151
        sma200    0.1092 0.1111 0.6437 -0.3296
        mix60     0.0851 0.0913 0.5182 -0.3400
        tbill     0.0394 0.0028    NaN  0.0000
val     bh        0.0821 0.1861 0.3190 -0.5525
        sma200    0.1019 0.1173 0.5722 -0.2013
        mix60     0.0670 0.1102 0.3092 -0.3704
        tbill     0.0375 0.0019    NaN  0.0000
holdout bh        0.1428 0.1715 0.7808 -0.3379
        sma200    0.0936 0.1263 0.6575 -0.2630
        mix60     0.0925 0.1017 0.7790 -0.2112
        tbill     0.0147 0.0014    NaN -0.0000
full    bh        0.1101 0.1645 0.5133 -0.5525
        sma200    0.1047 0.1154 0.6297 -0.3296
        mix60     0.0824 0.0977 0.5148 -0.3704
        tbill     0.0343 0.0025    NaN -0.0000

### Sharpe matrix (rows: period, cols: benchmark)

In [4]:
sharpe = tbl["Sharpe"].unstack("benchmark").reindex([p for p in bp.PERIOD_ORDER if p in tbl.index.get_level_values(0)])
display(sharpe)
sma_bt = bench.attrs["sma_bt"]
print("SMA200 switches (full):", int((sma_bt["pos"].diff().abs() > 0).sum()), "| time in equities:", f"{sma_bt['pos'].mean():.2%}")

benchmark,bh,mix60,sma200,tbill
period,,,,
dev,0.5110,0.5182,0.6437,NaN
val,0.3190,0.3092,0.5722,NaN
holdout,0.7808,0.7790,0.6575,NaN
full,0.5133,0.5148,0.6297,NaN


SMA200 switches (full): 136 | time in equities: 69.44%


## 3. Equity curves (log scale)

In [5]:
import plotly.graph_objects as go

eq = (1 + bench[["bh", "sma200", "mix60", "tbill"]].fillna(0)).cumprod()
fig = go.Figure()
for c in eq.columns:
    fig.add_trace(go.Scatter(x=eq.index, y=eq[c], name=c, mode="lines"))
for name in ("dev", "val", "holdout"):
    lo, hi = bp.period_bounds(name, eq.index)
    fig.add_vrect(x0=lo, x1=hi, opacity=0.06, line_width=0, annotation_text=name, annotation_position="top left")
fig.update_yaxes(type="log", title="growth of 1")
fig.update_layout(height=450, title="Benchmarks, S&P 500 TR vs T-bills, 1934 onward", legend_orientation="h")
fig.show()

## 4. Export reference figures for the registry

In [6]:
out = tbl[["CAGR", "Vol", "Sharpe", "Max DD"]].reset_index()
out_path = Path.cwd() / "benchmarks.csv"
out.to_csv(out_path, index=False, float_format="%.6f", lineterminator="\n")
print("written", out_path.name, out.shape)
display(out)

written benchmarks.csv (16, 6)


,period,benchmark,CAGR,Vol,Sharpe,Max DD
0,dev,bh,0.1106,0.1533,0.5110,-0.5151
1,dev,sma200,0.1092,0.1111,0.6437,-0.3296
2,dev,mix60,0.0851,0.0913,0.5182,-0.3400
3,dev,tbill,0.0394,0.0028,NaN,0.0000
4,val,bh,0.0821,0.1861,0.3190,-0.5525
5,val,sma200,0.1019,0.1173,0.5722,-0.2013
6,val,mix60,0.0670,0.1102,0.3092,-0.3704
7,val,tbill,0.0375,0.0019,NaN,0.0000
8,holdout,bh,0.1428,0.1715,0.7808,-0.3379
9,holdout,sma200,0.0936,0.1263,0.6575,-0.2630


## 5. Reading the table

* `sma200` is the bar to beat: a trial *passes* only if its Sharpe exceeds **both**
  `sma200` and `bh` in dev **and** val (bootstrap p <= 0.10 vs `sma200`).
* `mix60` shows how much of the SMA edge is simply lower equity exposure.
* The holdout column is reported for completeness; it does not enter the pass rule.